# Stickman Hybrid Studio — Kaggle GPU Worker (T4 x2)
1. Set **Accelerator**: `GPU T4 x2` & **Internet**: `On`.
2. Run Cell 1 to install dependencies & download `worker_server.py` from GitHub.
3. Run Cell 2 to start the FastAPI GPU server and print your **Cloudflare Tunnel URL**.
4. Paste the URL into your **Stickman Hybrid Studio** web app.

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate sentencepiece protobuf fastapi uvicorn pillow
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!wget -q https://raw.githubusercontent.com/lukman582890-wq/stickman_project/main/kaggle/worker_server.py -O worker_server.py
print('Setup complete!')

In [ ]:
import os, re, subprocess, time
os.environ['STICKMAN_API_KEY'] = ''  # Optional: set a secret key here

uvicorn_proc = subprocess.Popen(['uvicorn', 'worker_server:app', '--host', '0.0.0.0', '--port', '8000'])
time.sleep(3)
cf_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
for line in cf_proc.stdout:
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        print('\n========================================')
        print('KAGGLE WORKER URL:', m.group(0))
        print('========================================\n')
        break
cf_proc.wait()